# 02 — Vector Store & Retrieval
**Phase:** Foundation — Day 1

**What this notebook does:**
- Reopens the ChromaDB built in notebook 01
- Explores what is stored inside
- Experiments with different queries to understand retrieval
- Visualises distance scores

**Run notebook 01 before this.**

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

# Reopen — no re-embedding needed
client = chromadb.PersistentClient(path="../chroma_db")
collection = client.get_collection("my_docs")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

print(f"Collection: {collection.name}")
print(f"Total chunks stored: {collection.count()}")

## Step 1 — Peek inside the database
See what the first few stored entries look like.

In [ ]:
# Peek at first 3 entries
sample = collection.peek(limit=3)

for i in range(len(sample['ids'])):
    print(f"\n{'='*50}")
    print(f"ID      : {sample['ids'][i]}")
    print(f"Text    : {sample['documents'][i][:250]}")
    print(f"Embed[0]: {sample['embeddings'][i][:5]}  ... (384 total numbers)")

## Step 2 — Understanding distance scores
Run the same query multiple ways and compare scores.

In [ ]:
def retrieve(query, top_k=5):
    vec = embed_model.encode([query]).tolist()
    res = collection.query(query_embeddings=vec, n_results=top_k)
    return res

def show_results(query):
    res = retrieve(query)
    print(f"\nQuery: '{query}'")
    for i, (doc, cid, dist) in enumerate(
        zip(res['documents'][0], res['ids'][0], res['distances'][0])
    ):
        bar = '█' * int((1 - dist) * 20) if dist < 1 else ''
        quality = 'STRONG' if dist < 0.3 else 'GOOD' if dist < 0.6 else 'WEAK'
        print(f"  {cid:12} dist={dist:.4f} {bar:20} [{quality}]")
        print(f"  {doc[:120]}...\n")

# CHANGE THESE to match your document
show_results("What is the main topic?")
show_results("Something completely unrelated like pizza recipes")

## Step 3 — Semantic similarity experiment
Prove that semantic search finds meaning, not just keywords.

In [ ]:
# These two queries mean the same thing but use different words
# A keyword search would find different results.
# A semantic search should find SIMILAR results.

# CHANGE both queries to paraphrases of something in your document
query_a = "How does the model get trained?"      # direct phrasing
query_b = "What is the learning procedure?"      # paraphrase

res_a = retrieve(query_a)
res_b = retrieve(query_b)

ids_a = set(res_a['ids'][0])
ids_b = set(res_b['ids'][0])
overlap = ids_a & ids_b

print(f"Query A top chunks: {res_a['ids'][0]}")
print(f"Query B top chunks: {res_b['ids'][0]}")
print(f"\nOverlapping chunks: {overlap}")
print(f"Overlap count: {len(overlap)}/5")
print("\nIf overlap >= 3, semantic search is working correctly.")

## Key observations
- Distance < 0.3 = very strong match
- Distance 0.3–0.6 = good match  
- Distance > 0.6 = weak match — the answer may not be in your doc
- Semantic search finds paraphrases — keyword search cannot do this

**Next:** `03_groq_rag_pipeline.ipynb` — connect the LLM and generate answers.